<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T02-L01-autograd-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T02-L01-autograd-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T02-L01-autograd-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T02-L01-autograd-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T02-L01 · Autodiff from scratch: a scalar engine, a tensor engine, and the check that keeps both honest

**You will build:** reverse-mode automatic differentiation for scalars, then for numpy
tensors with broadcasting-aware gradients, a forward-mode engine to compare it with, and the
finite-difference check every gradient in this lesson is held to.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track.

No deep-learning framework is used anywhere in this lesson: only Python and numpy. An
optional cell near the end compares your engine with PyTorch, and runs only if PyTorch is
already installed on the machine you are using.

By the end you will be able to:

1. Implement a central-difference numerical gradient and a relative-error comparison, and use
   them as the oracle every other gradient in this lesson is checked against.
2. Implement reverse-mode autodiff for scalars: an iterative topological sort of the
   computation graph, and local backward rules that accumulate where a value is reused.
3. Implement forward-mode autodiff with dual numbers, and measure how many passes forward and
   reverse mode each need to produce a full gradient.
4. Implement broadcasting-aware tensor gradients (unbroadcast, sum, matmul) and a numerically
   stable softmax cross-entropy built from them, each verified to a stated tolerance.
5. Measure how the gradient norm changes from layer to layer in a deep network, for different
   activations and weight scales, using an engine you built yourself.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions and classes, including operator methods such as __add__ and __mul__, and pass a function as an argument to another function
- use numpy arrays: read a shape, pass an axis to sum, and use @ for a matrix product
- differentiate a product, a composition and tanh on paper with the chain rule

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [The spelled-out intro to neural networks and backpropagation: building micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) · Andrej Karpathy · *free* — a long video that builds a scalar reverse-mode engine live; it is the picture to have in mind for exercises 2 and 3
- [NumPy user guide: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · NumPy documentation · *free* — the shape-matching rules whose gradient exercise 5 undoes
- [CS231n: Neural Networks Part 3, Gradient checks](https://cs231n.github.io/neural-networks-3/) · Stanford CS231n course notes · *free* — why gradients are compared by relative error, and what size of error to worry about, as in exercise 1
- [Automatic Differentiation in Machine Learning: a Survey](https://jmlr.org/papers/v18/17-468.html) · Baydin, Pearlmutter, Radul and Siskind, JMLR · *free* — forward and reverse mode, dual numbers and the cost of each, for exercise 4 and the pass counts

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261002          # this lesson's one fixed seed: every random array below derives from
                          # it, so re-running prints the same numbers on any machine.
TOL = 1e-5               # the relative-error tolerance every gradient check is held to

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("numerical_gradient", "relative_error"),
    "exercise 2": ("topo_order",),
    "exercise 3": ("__mul__", "tanh"),
    "exercise 4": ("dual_mul", "dual_tanh", "jvp"),
    "exercise 5": ("unbroadcast",),
    "exercise 6": ("sum", "matmul"),
    "exercise 7": ("softmax_cross_entropy",),
    "exercise 8": ("layer_grad_norms",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (__mul__, tanh)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. A check still running after `_CHECK_SECONDS` (a demo: `_DEMO_SECONDS`) is stopped
    and counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check runs under a time limit far above what a right answer needs (the whole lesson runs in
# about a second); a demo gets a longer one.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 20


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str,
                why: str = "a loop that never reaches its stopping condition never ends",
                memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit, or an enclosing guard) keeps its deadline if it is due first and is put back,
    with the time it had left, afterwards. A correct answer finishes far inside the limit, and
    nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = f"{what} was still running after {seconds:g} s: {why}"
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])

## 1. Three ways to get a derivative, and the one that checks the others

Training a network means following the gradient of a loss with respect to every parameter.
There are three ways to get that gradient out of a program:

* **Numerically**, by nudging each input and watching the output move. It needs nothing but
  the function, so it cannot be wrong about the maths, but it costs two function evaluations
  per input and carries rounding error.
* **Symbolically**, by differentiating a formula on paper and typing the result in. It is
  exact, and it is what you do by hand for one network until you make a sign error.
* **Automatically**, by recording the elementary operations a program performs and applying
  the chain rule to the record. It is exact to rounding and its cost is a small multiple of
  the cost of the function itself.

Automatic differentiation is a family of techniques, not one algorithm: *reverse mode* is a
generalised backpropagation, and *forward mode* is its mirror image (see `claims.yaml` for
the sources). You will build both. But the first thing you build is the
slow, simple, hard-to-fool method: a numerical gradient. Every gradient this lesson ever
produces is compared with it, so a wrong backward rule shows up as a number, not a hunch.

In [ ]:
# One derivative, two ways, in plain Python: f(x) = x^3 at x = 2. Nothing of yours is used yet.
x0 = 2.0
on_paper = 3.0 * x0 * x0                  # the rule d/dx x^3 = 3x^2, applied by hand
print(f"derivative differentiated on paper: {on_paper}")
for eps in (1e-1, 1e-3, 1e-5):
    nudged = ((x0 + eps) * (x0 + eps) * (x0 + eps) - x0 * x0 * x0) / eps
    print(f"nudge x by {eps:.0e} and watch f move: {nudged:.6f}  (off by {abs(nudged - on_paper):.1e})")

## 2. Exercise 1 — the oracle: `numerical_gradient()` and `relative_error()`

The **central difference** estimates `df/dx` as `(f(x + h) - f(x - h)) / (2 * h)` for a
small `h` (`eps` below). Its error shrinks like `h` squared, where the one-sided
`(f(x + h) - f(x)) / h` shrinks only like `h`. `numerical_gradient` does this once per
element of an array, and must leave the array exactly as it found it.

Comparing two gradients needs a *relative* error, not a raw difference: a gap of `1e-4` is
nothing between gradients of size 1 and a disaster between gradients of size `1e-6`.

<details><summary>💡 Hint 1 — what to think about</summary>

Visit every element in turn, and for each one evaluate `f` twice with only that element
changed. Where do the two perturbed copies come from, and what does `f` see while the other
elements are being left alone? Two traps: an integer array cannot hold `x + 1e-6`, and an
array you perturb in place must be put back before you move on.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Work on a float64 copy. Walk its indices (`np.ndindex(x.shape)` works for any number of
dimensions, including none), remember the old value, write old plus eps, evaluate, write old
minus eps, evaluate, restore, divide the difference by twice eps. For the relative error,
take the absolute difference over the larger of the two magnitudes, element by element, then
the worst element. An element where both gradients are exactly zero is a perfect match, not
a division by zero.
</details>

In [ ]:
def numerical_gradient(f: Callable[[np.ndarray], float], x: np.ndarray,
                       eps: float = 1e-6) -> np.ndarray:
    """Central-difference gradient of the scalar function `f` at `x`.

    `f` takes an array of the same shape as `x` and returns a float. Return an array of the
    same shape as `x` whose element i is `(f(x + eps * e_i) - f(x - eps * e_i)) / (2 * eps)`,
    where e_i is the unit vector along element i. `x` itself must be left unchanged, and an
    integer-typed `x` must work (the answer is a float array either way).

    Example:
        >>> numerical_gradient(lambda a: float((a ** 2).sum()), np.array([1.0, 2.0]))
        # approximately array([2., 4.]), the gradient of sum(a**2) is 2a
    """
    # YOUR CODE HERE
    raise NotImplementedError


def relative_error(analytic: np.ndarray, numeric: np.ndarray) -> float:
    """Worst element-wise relative error between two gradients of the same shape.

    For each element the error is `|a - n| / max(|a|, |n|)`; an element where both are exactly
    zero has error 0. Return the largest of them as a Python float.

    Example:
        >>> relative_error(np.array([1.0, 0.0]), np.array([1.1, 0.0]))
        # approximately 0.0909: the first element gives 0.1 / 1.1, the second gives 0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_oracle() -> None:
    x = np.array([[1.0, -2.0], [3.0, 0.5]])
    before = x.copy()
    seen = []

    def f(a):
        seen.append(a.copy())
        return float((a ** 2).sum())

    g = numerical_gradient(f, x)
    assert isinstance(g, np.ndarray) and g.shape == x.shape, (
        "the gradient must be an array with the same shape as x"
    )
    assert np.allclose(g, 2 * x, rtol=0, atol=1e-6), (
        f"got {g.tolist()} but the gradient of sum(a**2) is 2a = {(2 * x).tolist()} -- "
        "divide the DIFFERENCE of the two evaluations by 2 * eps"
    )
    assert np.allclose(x, before, rtol=0, atol=1e-12), (
        "numerical_gradient changed the array you passed in -- perturb a copy, or put every "
        "element back to exactly its old value before moving to the next one"
    )
    for call in seen:
        moved = np.abs(call - before) > 1e-9
        assert moved.sum() == 1, (
            "f was called with more than one element changed: every evaluation must differ "
            "from x in exactly ONE element, so restore each element before the next"
        )
    cubic = numerical_gradient(lambda a: float(a[0] ** 3), np.array([2.0]), eps=1e-3)
    assert abs(cubic[0] - 12.0) < 1e-4, (
        f"got {cubic[0]!r} for d/dx x**3 at 2 with eps=1e-3 (exactly 12): that is the error "
        "of a ONE-sided difference; evaluate at x + eps AND x - eps"
    )
    ints = numerical_gradient(lambda a: float((a ** 2).sum()), np.array([1, 2, 3]))
    assert np.allclose(ints, [2.0, 4.0, 6.0], rtol=0, atol=1e-6), (
        f"got {ints.tolist()} for an integer input: an integer array cannot hold x + eps, so "
        "convert to float64 first"
    )
    scalar = numerical_gradient(lambda a: float(a * a), np.array(3.0))
    assert np.shape(scalar) == () and abs(float(scalar) - 6.0) < 1e-6, (
        "a 0-d input (a single number) must give a 0-d gradient"
    )
    one = relative_error(np.array([1.0, 0.0]), np.array([1.1, 0.0]))
    assert abs(one - 0.1 / 1.1) < 1e-12, (
        f"got {one!r}, expected |1.0 - 1.1| / max(1.0, 1.1) = 0.0909...: divide by the LARGER "
        "of the two magnitudes, not by one of them and not by their sum"
    )
    both_zero = relative_error(np.zeros(3), np.zeros(3))
    assert both_zero == 0.0, (
        f"got {both_zero!r} for two zero gradients: both zero is a perfect match, so return 0 "
        "rather than nan"
    )
    worst = relative_error(np.array([1.0, 1.0, 1.0, 1.0]), np.array([1.0, 1.0, 1.0, 2.0]))
    assert abs(worst - 0.5) < 1e-12, (
        f"got {worst!r}: the answer is the WORST element, not the mean"
    )
    mixed = relative_error(np.array([100.0, 1.0]), np.array([100.0, 2.0]))
    assert abs(mixed - 0.5) < 1e-12, (
        f"got {mixed!r}: the second element is off by a factor of two (error 0.5) next to a large "
        "element that matches; divide each element by ITS OWN larger magnitude, not by the "
        "largest magnitude in the whole array"
    )


_try("exercise 1", _check_oracle)

### The step size is a trade-off, and you can see it

Make `h` smaller and the truncation error falls; make it too small and the subtraction
`f(x + h) - f(x - h)` loses digits to rounding. The table below uses your oracle on a cubic
(built only from multiplications and subtractions, so the arithmetic is the same on every
machine) whose exact derivative is known, next to the one-sided difference.

In [ ]:
def _show_step_size() -> None:
    def cubic(a):
        return float(a[0] * a[0] * a[0] - 2.0 * a[0])

    x0 = np.array([1.5])
    exact = 3.0 * 1.5 * 1.5 - 2.0
    print(f"f(x) = x^3 - 2x at x = 1.5, exact derivative {exact}")
    print(f"{'eps':>8}  {'central error':>14}  {'one-sided error':>16}")
    central_errs, forward_errs = {}, {}
    for k in (1, 3, 5, 7, 9, 11, 13):
        eps = 10.0 ** (-k)
        central = float(numerical_gradient(cubic, x0, eps)[0])
        forward = (cubic(x0 + eps) - cubic(x0)) / eps
        central_errs[eps], forward_errs[eps] = abs(central - exact), abs(forward - exact)
        print(f"{eps:8.0e}  {central_errs[eps]:14.1e}  {forward_errs[eps]:16.1e}")
    best_c = min(central_errs, key=central_errs.get)
    best_f = min(forward_errs, key=forward_errs.get)
    print(f"\nsmallest error in this table: central at eps={best_c:.0e} "
          f"({central_errs[best_c]:.1e}), one-sided at eps={best_f:.0e} "
          f"({forward_errs[best_f]:.1e})")
    steps = list(central_errs)                     # eps from largest to smallest

    def worse_either_side(errs: dict) -> bool:
        i = steps.index(min(errs, key=errs.get))
        return all(errs[steps[j]] > errs[steps[i]] for j in (i - 1, i + 1) if 0 <= j < len(steps))

    print(f"best central error is {central_errs[best_c] / forward_errs[best_f]:.0e} of the best "
          f"one-sided error")
    print("one step either side of its best eps, each difference has a larger error: "
          + ("yes, truncation on the larger-step side and rounding on the smaller"
             if worse_either_side(central_errs) and worse_either_side(forward_errs) else "no"))


_try("step-size demo", _show_step_size, needs=("exercise 1",))

## 3. The computation graph, and Exercise 2 — `topo_order()`

Reverse mode starts by recording what happened. Every operation produces a new node that
remembers its inputs (`_prev`) and a small function (`_backward`) that knows how to push a
gradient from the node back to those inputs. Add a `*` and a `tanh` and the nodes form a
directed acyclic graph: a node may feed several others (a *diamond*), which is why a value
used twice collects gradient from both uses.

To run every `_backward` in the right order, a node must be processed only after every node
that consumed it. Reverse a list in which every node appears **after** its inputs and that
is exactly the order you need. You will write the function that builds the list.

A tempting recursive depth-first search works on a toy graph and then raises
`RecursionError` on a long chain, because Python limits how deep a recursion can go. A graph
recorded from a loop of a few thousand operations is entirely ordinary.

<details><summary>💡 Hint 1 — what to think about</summary>

Each node must appear exactly once, even if many nodes consume it, and after everything it
was computed from. How can you remember which nodes you have already visited when two
different Python objects may hold equal numbers? And what replaces the call stack if you do
not recurse?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep an explicit stack of (node, expanded) pairs. Pop one; if it is expanded, append it to
the output list. Otherwise, if its `id` is new, remember it, push it back marked expanded,
and push its `_prev` nodes unexpanded. Track visited nodes by `id(node)`, never by value.
</details>

In [ ]:
def topo_order(root: Any) -> list:
    """Every node reachable from `root` through `_prev`, each exactly once, ordered so that a
    node comes AFTER all the nodes in its `_prev`. `root` is therefore last.

    Nodes are objects with a `_prev` tuple (a node used twice by one operation, as in `x * x`,
    lists it twice). Must not recurse: a chain of 10,000 operations has to work.

    Example:
        a = Value(1.0); b = a * a; c = b + a        # c._prev == (b, a), b._prev == (a, a)
        topo_order(c)  ->  [a, b, c]
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _Node:
    """A bare graph node for testing topo_order on its own, before any maths exists.

    It counts how often its inputs are read and raises once that is absurd, so a walk that
    forgets which nodes it has already seen fails with a message instead of running for hours.
    """
    reads = 0
    limit = 10 ** 12

    def __init__(self, *prev):
        self._p = tuple(prev)

    @property
    def _prev(self):
        _Node.reads += 1
        if _Node.reads > _Node.limit:
            raise AssertionError(
                "topo_order read the inputs of nodes far more often than there are nodes: a "
                "node reached by many paths must be visited once, so remember which nodes "
                "you have already seen"
            )
        return self._p


def _check_topo_order() -> None:
    _Node.reads, _Node.limit = 0, 10 ** 12
    a = _Node(); b = _Node(a, a); c = _Node(b, a)
    _Node.reads, _Node.limit = 0, 100
    with _time_limit(1.0, "topo_order on a 3-node graph",
                     "a loop whose stack never shrinks never ends (read the top with "
                     "stack.pop(), not stack[-1])"):
        order = topo_order(c)
    assert [id(n) for n in order] == [id(a), id(b), id(c)], (
        f"got {len(order)} nodes for a 3-node graph (a, b = f(a, a), c = f(b, a)): each node "
        "must appear exactly once, inputs before consumers, root last"
    )
    _Node.reads, _Node.limit = 0, 10 ** 12
    depth = 4 * sys.getrecursionlimit()
    root = _Node()
    for _ in range(depth):
        root = _Node(root)
    _Node.reads, _Node.limit = 0, 10 * depth
    try:
        with _time_limit(5.0, f"topo_order on a chain of {depth + 1} nodes"):
            deep = topo_order(root)
    except RecursionError:
        raise AssertionError(
            "a chain deeper than Python's recursion limit raised RecursionError: use an "
            "explicit stack of (node, expanded) pairs instead of recursion"
        ) from None
    assert len(deep) == depth + 1 and deep[-1] is root, (
        "a chain deeper than Python's recursion limit must still be ordered: use an explicit "
        "stack, not recursion"
    )
    _Node.reads, _Node.limit = 0, 10 ** 12
    layer = _Node()
    for _ in range(30):
        layer = _Node(layer, layer)           # 30 diamonds: 2**30 paths, only 31 nodes
    _Node.reads, _Node.limit = 0, 10 * 31
    with _time_limit(1.0, "topo_order on a 31-node ladder of diamonds"):
        laddered = topo_order(layer)
    assert len(laddered) == 31, (
        "a node reached by many paths must appear once: remember visited ids"
    )
    _Node.limit = 10 ** 12


_try("exercise 2", _check_topo_order)

## 4. Exercise 3 — a scalar engine: `Value.__mul__` and `Value.tanh`

`Value` below is a complete scalar node with `+`, `-`, `**`, `exp`, `relu` and division
already written, so you can read how a local backward rule looks. Two are missing.

* `a * b`: the derivative of the product with respect to `a` is `b`, and with respect to
  `b` is `a`. Multiply by the gradient arriving from the output (the chain rule).
* `tanh`: its derivative is `1 - tanh(x)**2`, which you can compute from the output.

Read how the finished `__add__` accumulates with `+=`. The reason is the diamond from the
last section: in `x * x + x` the node `x` receives gradient from three places. Assigning
with `=` instead of `+=` keeps only the last one, and the result looks plausible until you
check it numerically, which the cell after this exercise does on purpose.

<details><summary>💡 Hint 1 — what to think about</summary>

The new node's `_prev` must contain both operands, or `topo_order` will never find them. A
plain Python number on either side must work too: wrap it in a `Value` the way `__add__`
does. What must happen when `self` and `other` are the same object?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Inside `__mul__`, define a `_backward` that hands each operand the OTHER operand's value
times the gradient at the output, added to what the operand already holds; attach it to the
new node and return the node. For `tanh`, compute the output value first, because the local
derivative is written in terms of it; then, again, multiply by the gradient at the output
and add.
</details>

In [ ]:
class Value:
    """A scalar in a computation graph: its number, its gradient, and how it was made."""

    def __init__(self, data: float, _prev: tuple = (), _op: str = ""):
        self.data = float(data)
        self.grad = 0.0
        self._backward: Callable[[], None] = lambda: None
        self._prev = tuple(_prev)
        self._op = _op

    def __repr__(self) -> str:
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        """`self * other`: a new Value whose backward rule feeds both operands.

        Wrap a plain number in a Value, record both operands in `_prev`, and make `_backward`
        ADD each operand's share of `out.grad` to its `grad`.

        Example:
            >>> a, b = Value(2.0), Value(3.0)
            >>> c = a * b; c.backward()
            >>> a.grad, b.grad            # dc/da = b, dc/db = a
            (3.0, 2.0)
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def tanh(self):
        """Hyperbolic tangent. Local derivative: `1 - tanh(x) ** 2`.

        Example:
            >>> x = Value(0.0)
            >>> y = x.tanh(); y.backward()
            >>> x.grad                    # 1 - tanh(0)**2
            1.0
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only a constant exponent is supported"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0.0 else 0.0, (self,), "relu")

        def _backward():
            self.grad += (1.0 if out.data > 0.0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other if isinstance(other, Value) else Value(-other))

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return Value(other) + (-self)

    def __truediv__(self, other):
        return self * (other if isinstance(other, Value) else Value(other)) ** -1

    def __rtruediv__(self, other):
        return Value(other) * self ** -1

    def backward(self) -> None:
        """Reverse-mode sweep from this node: its own gradient is 1, then every node's
        `_backward` runs once, consumers before the nodes they consumed. The gradient of every
        node in this graph is reset first, so calling `backward` twice gives the same answer as
        once; a node that is not in this graph keeps whatever gradient it had."""
        order = topo_order(self)
        for node in order:
            node.grad = 0.0
        self.grad = 1.0
        for node in reversed(order):
            node._backward()


def _check_value() -> None:
    a, b = Value(2.0), Value(3.0)
    c = a * b
    assert a in c._prev and b in c._prev, (
        "the node a * b must list BOTH operands in _prev, or the sweep cannot find them"
    )
    c.backward()
    assert (c.data, a.grad, b.grad) == (6.0, 3.0, 2.0), (
        f"got value {c.data}, da={a.grad}, db={b.grad}; for c = a * b expect 6.0, 3.0, 2.0: "
        "da is the OTHER operand times the gradient arriving at c"
    )
    x = Value(3.0)
    y = x * x + x
    y.backward()
    assert x.grad == 7.0, (
        f"d/dx (x*x + x) at 3 is 2x + 1 = 7 but got {x.grad}: x is used three times, so each "
        "use must ADD to x.grad (+=), never overwrite it"
    )
    z = Value(2.0)
    w = 3.0 * z * 4.0
    w.backward()
    assert (w.data, z.grad) == (24.0, 12.0), (
        f"got value {w.data}, grad {z.grad} for 3.0 * z * 4.0 at z=2: a plain number on either "
        "side of * must be wrapped in a Value"
    )
    t = Value(0.5)
    s = t.tanh()
    s.backward()
    assert abs(s.data - math.tanh(0.5)) < 1e-12 and abs(t.grad - (1 - math.tanh(0.5) ** 2)) < 1e-12, (
        f"tanh(0.5) has derivative 1 - tanh(0.5)**2 = {1 - math.tanh(0.5) ** 2:.6f}, got {t.grad}: "
        "square the OUTPUT of tanh, subtract from 1"
    )
    p, q = Value(0.7), Value(-1.2)
    r = (p * q + 0.3).tanh() * p
    r.backward()
    want_p = (1 - math.tanh(0.7 * -1.2 + 0.3) ** 2) * (-1.2) * 0.7 + math.tanh(0.7 * -1.2 + 0.3)
    assert abs(p.grad - want_p) < 1e-10, (
        f"got {p.grad}, expected {want_p}: p is used in two places, and the chain rule must "
        "run through tanh into the product"
    )


_try("exercise 3", _check_value, needs=("exercise 2",))

### Check it against the oracle, and watch it catch a bug

`value_gradient_error` runs your engine and your oracle on the same function and reports
the worst relative error. The second half of the cell does the same for a deliberately
broken multiplication (it overwrites the gradient instead of adding), so you can see what a
genuine backward-rule bug looks like next to a correct engine.

In [ ]:
def value_gradient_error(fn: Callable[[list], "Value"], xs: Sequence[float],
                         eps: float = 1e-5) -> float:
    """Worst relative error between `Value.backward` and a numerical gradient of `fn`."""
    leaves = [Value(x) for x in xs]
    fn(leaves).backward()
    analytic = np.array([v.grad for v in leaves])
    numeric = numerical_gradient(lambda a: fn([Value(float(t)) for t in a]).data,
                                 np.array(xs, dtype=np.float64), eps)
    return relative_error(analytic, numeric)


class _BuggyValue(Value):
    """Value with one wrong backward rule: `=` where there should be `+=`."""
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        return out


def _expr(leaves):
    x, y = leaves
    return (x * y + x * x).tanh() + y * 0.5


def _show_value_check() -> None:
    xs = [0.8, -0.4]
    good = value_gradient_error(_expr, xs)
    # Below 1e-8 the error is finite-difference rounding noise whose last digits differ from one
    # machine's floating-point library to another's, so it prints as a bound, not as digits.
    shown = "below 1e-08 (rounding noise)" if good < 1e-8 else f"{good:.1e}"
    print(f"your engine: worst relative error {shown}  "
          f"{'within' if good <= TOL else 'ABOVE'} the tolerance {TOL:.0e}")

    x, y = _BuggyValue(xs[0]), _BuggyValue(xs[1])
    out = (x * y + x * x).tanh() + y * 0.5
    out.backward()
    analytic = np.array([x.grad, y.grad])
    numeric = numerical_gradient(lambda a: _expr([Value(float(t)) for t in a]).data,
                                 np.array(xs), 1e-5)
    bad = relative_error(analytic, numeric)
    print(f"buggy multiply (assigns, never accumulates): worst relative error {bad:.1e}  "
          f"{'within' if bad <= TOL else 'ABOVE'} the tolerance {TOL:.0e}")
    print(f"d/dx by the buggy engine {analytic[0]:.4f}, by the oracle {numeric[0]:.4f}: "
          "the bug is not subtle once something is measuring it")


_try("value check demo", _show_value_check, needs=("exercise 1", "exercise 2", "exercise 3"))

## 5. Exercise 4 — forward mode: `dual_mul()`, `dual_tanh()` and `jvp()`

Reverse mode walks the graph from the output back to the inputs. **Forward mode** walks the
other way: every number carries a second number alongside it, its *tangent*, which says how
fast that number changes if you move the inputs in one chosen direction. Pair them up as a
`Dual(val, tan)` and the rules are the ones you already know:

* sum: tangents add, and the product rule applies to `a * b`;
* `tanh`: the tangent is multiplied by the derivative of `tanh`, evaluated at `val`.

One forward pass with the tangents set to a direction `v` returns the derivative of the whole
function along `v`: a **Jacobian-vector product**, or JVP. (Reverse mode's one pass returns
the transpose-direction product, a VJP. For a function with one output, the VJP with weight
1 is the whole gradient.) `Dual` below has `+`, `-`, `exp` and the operator plumbing already;
you write the two rules and the one-pass function. Its scope is deliberately small: scalar
inputs, one scalar output, and one direction per pass (no batch of tangents, no Jacobian of a
vector-valued function). PyTorch ships forward mode too, still marked beta in its
documentation as of 2 October 2026.

<details><summary>💡 Hint 1 — what to think about</summary>

For `dual_mul`, a product's rate of change has two terms, one for each factor moving while
the other stays put. For `jvp`, what are the tangents of the inputs, and what do you read
off the output at the end?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

A product's value is the product of the values; its tangent is each factor's tangent times
the other factor's value, added together. The value of a tanh is the tanh of the value; its
tangent is the incoming tangent times the derivative of tanh at that value. `jvp` wraps each
input in a `Dual` whose tangent is the matching component of the direction, calls `f` once,
and returns the output's value and tangent as a pair.
</details>

In [ ]:
def dual_mul(a: "Dual", b: "Dual") -> "Dual":
    """Product of two dual numbers: the product rule on the tangent.

    Example:
        >>> c = dual_mul(Dual(2.0, 1.0), Dual(3.0, 0.5))
        >>> (c.val, c.tan)                # 2*3 and 1*3 + 2*0.5
        (6.0, 4.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def dual_tanh(a: "Dual") -> "Dual":
    """tanh of a dual number: the chain rule on the tangent.

    Example:
        >>> d = dual_tanh(Dual(0.0, 2.0))
        >>> (d.val, d.tan)                # tanh(0) and (1 - 0**2) * 2
        (0.0, 2.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def jvp(f: Callable[[list], "Dual"], xs: Sequence[float], v: Sequence[float]) -> tuple:
    """One forward pass: `(f(xs), derivative of f at xs along the direction v)`.

    `f` takes a list of Dual and returns one Dual. Call it exactly once.

    Example:
        >>> jvp(lambda d: d[0] * d[1], [2.0, 3.0], [1.0, 0.0])
        (6.0, 3.0)        # the value, and d/dx0 of x0*x1 = x1
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _as_dual(x) -> "Dual":
    return x if isinstance(x, Dual) else Dual(x)


class Dual:
    """A number and its tangent: how fast it moves when the inputs move along one direction."""

    def __init__(self, val: float, tan: float = 0.0):
        self.val = float(val)
        self.tan = float(tan)

    def __repr__(self) -> str:
        return f"Dual(val={self.val:.4f}, tan={self.tan:.4f})"

    def __add__(self, other):
        other = _as_dual(other)
        return Dual(self.val + other.val, self.tan + other.tan)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return Dual(-self.val, -self.tan)

    def __sub__(self, other):
        return self + (-_as_dual(other))

    def __mul__(self, other):
        return dual_mul(self, _as_dual(other))

    def __rmul__(self, other):
        return dual_mul(_as_dual(other), self)

    def tanh(self):
        return dual_tanh(self)

    def exp(self):
        e = math.exp(self.val)
        return Dual(e, e * self.tan)


def forward_gradient(f: Callable[[list], "Dual"], xs: Sequence[float]) -> np.ndarray:
    """The full gradient by forward mode: one JVP per input, each along a unit direction."""
    n = len(xs)
    return np.array([jvp(f, xs, [1.0 if j == i else 0.0 for j in range(n)])[1]
                     for i in range(n)])


def _check_dual() -> None:
    c = Dual(2.0, 1.0) * Dual(3.0, 0.5)
    assert (c.val, c.tan) == (6.0, 4.0), (
        f"got ({c.val}, {c.tan}) for (2, tangent 1) * (3, tangent 0.5), expected (6.0, 4.0): "
        "the product rule is a.tan * b.val + a.val * b.tan"
    )
    d = Dual(0.5, 2.0).tanh()
    want = 2.0 * (1.0 - math.tanh(0.5) ** 2)
    assert abs(d.val - math.tanh(0.5)) < 1e-12 and abs(d.tan - want) < 1e-12, (
        f"got {d!r}, expected tangent {want:.6f}: multiply the incoming tangent by "
        "1 - tanh(val)**2"
    )

    def f(d):
        return d[0] * d[1] + d[0].tanh()
    xs = [0.5, 2.0]
    value, along_x0 = jvp(f, xs, [1.0, 0.0])
    _, along_x1 = jvp(f, xs, [0.0, 1.0])
    _, along_both = jvp(f, xs, [1.0, 1.0])
    assert abs(value - (0.5 * 2.0 + math.tanh(0.5))) < 1e-12, (
        f"got value {value}: jvp returns f(xs) itself as its first element"
    )
    assert abs(along_x0 - (2.0 + 1 - math.tanh(0.5) ** 2)) < 1e-12 and abs(along_x1 - 0.5) < 1e-12, (
        f"got derivatives {along_x0} and {along_x1} along the two unit directions: seed the "
        "tangent of input i with v[i], not with 1"
    )
    assert abs(along_both - (along_x0 + along_x1)) < 1e-12, (
        "the derivative along (1, 1) must be the sum of the derivatives along (1, 0) and (0, 1)"
    )
    calls = []
    jvp(lambda d: calls.append(1) or d[0], [1.0], [1.0])
    assert len(calls) == 1, f"jvp called f {len(calls)} times; it is ONE forward pass"


_try("exercise 4", _check_dual)

### Forward against reverse: count the passes

For a function of `n` inputs and one output, forward mode returns one directional derivative
per pass, so the full gradient costs `n` passes. Reverse mode returns the whole gradient from
one backward sweep over the graph recorded by one forward evaluation. The cell below runs
both on the same many-input function with a counter on `f` and checks that they agree. The
mirror case, one input and many outputs, is where forward mode wins: one pass gives every
output's derivative (see `claims.yaml`). This lesson does not build that case.

In [ ]:
def _wide(xs):
    """40 inputs in a ring: sum of tanh(x_i * x_{i+1} + x_i / 2). One scalar output."""
    n = len(xs)
    total = 0.0
    for i in range(n):
        total = total + (xs[i] * xs[(i + 1) % n] + 0.5 * xs[i]).tanh()
    return total


def _show_forward_vs_reverse() -> None:
    xs = list(np.random.default_rng(SEED).uniform(-1.0, 1.0, size=40))
    n_evals = [0]

    def counted(inputs):
        n_evals[0] += 1
        return _wide(inputs)

    fwd = forward_gradient(counted, xs)
    fwd_evals = n_evals[0]
    n_evals[0] = 0
    leaves = [Value(x) for x in xs]
    out = counted(leaves)
    out.backward()
    rev = np.array([v.grad for v in leaves])
    print(f"inputs: {len(xs)}")
    print(f"forward mode: {fwd_evals} evaluations of f (one JVP per input) for the full gradient")
    print(f"reverse mode: {n_evals[0]} evaluation of f, then 1 backward sweep over "
          f"{len(topo_order(out))} recorded nodes")
    err = relative_error(rev, fwd)
    print(f"the two gradients agree to a relative error below 1e-9: "
          f"{'yes' if err < 1e-9 else 'NO'}")
    print(f"for this function, {len(xs)} inputs and one output, reverse mode needed "
          f"{fwd_evals // max(n_evals[0], 1)}x fewer evaluations of f than forward mode")


_try("forward vs reverse demo", _show_forward_vs_reverse,
     needs=("exercise 2", "exercise 3", "exercise 4"))

## 6. Broadcasting, and Exercise 5 — `unbroadcast()`

Real networks add a bias vector of shape `(4,)` to a batch of shape `(3, 4)`. Numpy
**broadcasts**: it compares shapes from the trailing dimension leftwards, and two dimensions
are compatible when they are equal or one of them is 1 (see `claims.yaml`). A missing leading
dimension is treated as 1, and a size-1 dimension is stretched.

The forward pass therefore *copies* the bias onto each row. The backward pass must undo that:
the bias was used three times, so it collects the gradient from all three, which means
**summing** the incoming gradient over every axis that was added or stretched. If you do not,
the gradient arrives with the wrong shape, or, worse, with the right shape and wrong values.

<details><summary>💡 Hint 1 — what to think about</summary>

There are two different ways a dimension can have been broadcast, and they leave different
marks on the shapes: dimensions that do not exist in the target shape at all, and dimensions
that exist but equal 1. A size-1 dimension you sum away has to keep its place.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

First sum over axis 0 for as long as `grad` has more dimensions than `shape`. Then for every
axis where `shape` has size 1 but `grad` does not, sum over that axis keeping the dimension
(`keepdims=True`). The result must have exactly the requested shape.
</details>

In [ ]:
def unbroadcast(grad: np.ndarray, shape: tuple) -> np.ndarray:
    """Sum `grad` down to `shape`, undoing numpy broadcasting. `shape` must be one that
    broadcasts to `grad.shape`. The result has exactly `shape`.

    Example:
        >>> unbroadcast(np.ones((3, 4)), (4,))       # a (4,) bias used on 3 rows
        array([3., 3., 3., 3.])
        >>> unbroadcast(np.ones((3, 4)), (3, 1)).shape
        (3, 1)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_unbroadcast() -> None:
    g = np.arange(12, dtype=float).reshape(3, 4)
    with _time_limit(1.0, "unbroadcast of a (3, 4) gradient to (4,)",
                     "a loop that sums the leading axis with keepdims=True never reduces the "
                     "number of dimensions, so it never ends"):
        out = unbroadcast(g, (4,))
    assert out.shape == (4,) and np.array_equal(out, g.sum(axis=0)), (
        f"got shape {out.shape}, values {out.tolist()}: a (4,) operand used on 3 rows collects "
        "the SUM of the 3 row gradients"
    )
    out = unbroadcast(g, (3, 1))
    assert out.shape == (3, 1) and np.array_equal(out[:, 0], g.sum(axis=1)), (
        f"got shape {out.shape} for target (3, 1): sum over the stretched axis but keep it "
        "(keepdims=True), so the shape matches"
    )
    out = unbroadcast(g, ())
    assert np.shape(out) == () and float(out) == g.sum(), (
        "a scalar operand collects the sum of EVERY element of the gradient"
    )
    same = unbroadcast(g, (3, 4))
    assert same.shape == (3, 4) and np.array_equal(same, g), (
        "a gradient that already has the target shape must come back unchanged"
    )
    keep = unbroadcast(np.ones((1, 4)), (1, 4))
    assert keep.shape == (1, 4), (
        f"got shape {keep.shape}: a size-1 dimension that was NOT stretched must stay as it is"
    )
    lead = unbroadcast(np.ones((2, 3, 4)), (3, 1))
    assert lead.shape == (3, 1) and np.all(lead == 8.0), (
        f"got shape {lead.shape}, values {np.unique(lead).tolist()}: sum the missing leading "
        "axis AND the stretched one"
    )


_try("exercise 5", _check_unbroadcast)

### What goes wrong without it

The cell below adds a `(4,)` bias to a `(3, 4)` batch, pushes a gradient of ones back through
the addition, and prints the shape and the values of the bias gradient with and without your
`unbroadcast`. Every number is computed from the arrays.

In [ ]:
def _show_unbroadcast() -> None:
    batch_shape, bias_shape = (3, 4), (4,)
    upstream = np.ones(batch_shape)
    naive = upstream                      # what a backward rule that ignores broadcasting hands back
    proper = unbroadcast(upstream, bias_shape)
    print(f"batch {batch_shape}, bias {bias_shape}; the bias is used once per row, "
          f"{batch_shape[0]} times")
    print(f"without unbroadcast: gradient of shape {naive.shape}, which cannot even be added to a "
          f"bias of shape {bias_shape}")
    print(f"with unbroadcast:    gradient of shape {proper.shape}, every entry {proper[0]:.0f} "
          f"(= the {batch_shape[0]} rows it was copied onto)")
    wrong_mean = naive.mean(axis=0)
    print(f"averaging instead of summing would give {wrong_mean[0]:.0f} per entry: the loss "
          f"really does depend on each copy of the bias, so the copies' gradients ADD")


_try("unbroadcast demo", _show_unbroadcast, needs=("exercise 5",))

## 7. Exercise 6 — a tensor engine: `Tensor.sum` and `Tensor.matmul`

`Tensor` is the same idea as `Value`, but a node holds a numpy array and its gradient is an
array of the same shape. The elementwise operations are written for you (each one uses
`unbroadcast` from exercise 5 so a broadcast operand gets a gradient of its own shape). Two
operations are yours, and between them they cover most of what a layer does.

* `sum(axis, keepdims)`: the forward pass reduces, so the backward pass must *spread* the
  incoming gradient back over the axis that was summed. After `x.sum(axis=1)` the gradient
  has one fewer dimension than `x`; put the dimension back before broadcasting it.
* `matmul`: for `C = A @ B`, the gradient with respect to `A` is `dC @ B.T` and with respect
  to `B` is `A.T @ dC`. Check the shapes: they are the only way to remember which is which.

Test both on a square matrix and a non-square one. A square matrix hides a transposed or
misaligned gradient, because the shapes still fit.

<details><summary>💡 Hint 1 — what to think about</summary>

For `sum`: what shape does `out.grad` have when `axis` is given and `keepdims` is False, and
what shape does `self.grad` need? Which numpy function puts a dimension back? For `matmul`:
write the shapes of A, B and C down and see which two products of `dC` with the other
operand have the shape of A and of B.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

In `sum`'s backward, when an axis was given and `keepdims` is False, reinsert the summed
axes into the incoming gradient (numpy has a function for that, and it accepts a tuple of
axes), then broadcast the result to the shape of `self.data` and add it to `self.grad`. In
`matmul`'s backward, each operand's gradient is a product of the incoming gradient with the
other operand, one of the two transposed so that the shapes work out; add both.
</details>

In [ ]:
def _as_tensor(x) -> "Tensor":
    return x if isinstance(x, Tensor) else Tensor(x)


class Tensor:
    """An array in a computation graph: its data, a gradient of the same shape, how it was made."""

    def __init__(self, data, _prev: tuple = (), _op: str = ""):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward: Callable[[], None] = lambda: None
        self._prev = tuple(_prev)
        self._op = _op

    @property
    def shape(self) -> tuple:
        return self.data.shape

    def __repr__(self) -> str:
        return f"Tensor(shape={self.data.shape})"

    def __add__(self, other):
        other = _as_tensor(other)
        out = Tensor(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += unbroadcast(out.grad, self.data.shape)
            other.grad += unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = _as_tensor(other)
        out = Tensor(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += unbroadcast(other.data * out.grad, self.data.shape)
            other.grad += unbroadcast(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only a constant exponent is supported"
        out = Tensor(self.data ** k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Tensor(np.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Tensor(np.log(self.data), (self,), "log")

        def _backward():
            self.grad += out.grad / self.data
        out._backward = _backward
        return out

    def tanh(self):
        out = Tensor(np.tanh(self.data), (self,), "tanh")

        def _backward():
            self.grad += (1.0 - out.data ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(self.data, 0.0), (self,), "relu")

        def _backward():
            self.grad += (out.data > 0.0) * out.grad
        out._backward = _backward
        return out

    @property
    def T(self):
        out = Tensor(self.data.T, (self,), "T")

        def _backward():
            self.grad += out.grad.T
        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims: bool = False):
        """Sum over `axis` (an int, a tuple of ints, or None for everything).

        Backward: the gradient of every summed element is the gradient of the sum, so spread
        `out.grad` back over the axes that were reduced. When `keepdims` is False those axes
        are missing from `out.grad`; put them back before broadcasting.

        Example:
            >>> x = Tensor(np.ones((2, 3))); y = x.sum(axis=1)     # shape (2,)
            >>> y.grad = np.array([1.0, 2.0]); y._backward()
            >>> x.grad                    # each row receives its own row's gradient
            array([[1., 1., 1.], [2., 2., 2.]])
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def matmul(self, other):
        """Matrix product of two 2-D tensors, `(n, k) @ (k, m) -> (n, m)`.

        Backward: `dA = dC @ B.T` and `dB = A.T @ dC`, both ADDED to the existing gradients.

        Example:
            >>> a, b = Tensor(np.ones((2, 3))), Tensor(np.ones((3, 4)))
            >>> c = a.matmul(b); c.grad = np.ones((2, 4)); c._backward()
            >>> a.grad.shape, b.grad.shape
            ((2, 3), (3, 4))
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __matmul__(self, other):
        return self.matmul(other)

    def mean(self, axis=None, keepdims: bool = False):
        n = self.data.size if axis is None else int(
            np.prod([self.data.shape[a] for a in np.atleast_1d(axis)]))
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-_as_tensor(other))

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return _as_tensor(other) + (-self)

    def __truediv__(self, other):
        return self * _as_tensor(other) ** -1

    def backward(self) -> None:
        """Reverse-mode sweep from this node. Its gradient is set to ones (the gradient of the
        sum of its elements), the gradient of every node in this graph is reset first (a node
        outside it keeps whatever it had), then every `_backward` runs once, consumers before
        the nodes they consumed."""
        order = topo_order(self)
        for node in order:
            node.grad = np.zeros_like(node.data)
        self.grad = np.ones_like(self.data)
        for node in reversed(order):
            node._backward()


def gradient_error(build: Callable[..., "Tensor"], arrays: Sequence[np.ndarray],
                   eps: float = 1e-5) -> float:
    """Worst relative error between `Tensor.backward` and a numerical gradient, over every
    input array. `build` takes one Tensor per array and returns a scalar (0-d) Tensor."""
    tensors = [Tensor(a) for a in arrays]
    build(*tensors).backward()
    worst = 0.0
    for i, t in enumerate(tensors):
        def f(a, i=i):
            args = [Tensor(b) for b in arrays]
            args[i] = Tensor(a)
            return float(build(*args).data)
        worst = max(worst, relative_error(t.grad, numerical_gradient(f, arrays[i], eps)))
    return worst


def _check_tensor() -> None:
    rng = np.random.default_rng(SEED)
    sq, sq2 = rng.standard_normal((3, 3)), rng.standard_normal((3, 3))
    w_sq = Tensor(rng.standard_normal((3, 3)))
    err = gradient_error(lambda a, b: ((a @ b) * w_sq).sum(), [sq, sq2])
    assert err < TOL, (
        f"matmul gradient on square matrices: worst relative error {err:.1e}, tolerance "
        f"{TOL:.0e}. dA = dC @ B.T and dB = A.T @ dC: transpose the OTHER operand"
    )
    a, b = rng.standard_normal((2, 3)), rng.standard_normal((3, 4))
    w = Tensor(rng.standard_normal((2, 4)))
    err = gradient_error(lambda x, y: ((x @ y) * w).sum(), [a, b])
    assert err < TOL, (
        f"matmul gradient on a (2,3) @ (3,4) product: worst relative error {err:.1e}; the "
        "gradient of each operand must have that operand's own shape"
    )
    x = rng.standard_normal((3, 3))
    for axis in (0, 1):
        weights = Tensor(rng.standard_normal(3))
        err = gradient_error(lambda t, axis=axis, weights=weights: (t.sum(axis=axis) * weights).sum(), [x])
        assert err < TOL, (
            f"sum(axis={axis}) on a SQUARE matrix: worst relative error {err:.1e}. Shapes fit "
            "either way, so a gradient spread along the wrong axis passes silently: put the "
            "reduced axis back with np.expand_dims before broadcasting"
        )
    col_w = Tensor(rng.standard_normal((3, 1)))
    err = gradient_error(lambda t: (t.sum(axis=1, keepdims=True) * col_w).sum(), [x])
    assert err < TOL, f"sum(axis=1, keepdims=True): worst relative error {err:.1e}"
    err = gradient_error(lambda t: (t * t).sum(), [x])
    assert err < TOL, f"sum over every element: worst relative error {err:.1e}"
    bias = rng.standard_normal(4)
    batch = rng.standard_normal((3, 4))
    err = gradient_error(lambda p, q: ((p + q) * (p + q)).sum(), [batch, bias])
    assert err < TOL, (
        f"a (3,4) batch plus a (4,) bias: worst relative error {err:.1e}; the bias gradient "
        "needs unbroadcast (exercise 5), and the reduction needs sum (this exercise)"
    )


_try("exercise 6", _check_tensor, needs=("exercise 1", "exercise 2", "exercise 5"))

### Why tensors at all? Count the nodes

The scalar engine is easy to read because every multiplication is its own node. That is also
why it does not scale: one matrix product of two `n x n` matrices is `n**3` multiplications.
The cell below builds the same loss, the sum of a matrix product, in both engines, checks that
the two gradients agree, and prints how many nodes each recorded and how long each took.
Timings are this machine's; the node counts are exact.

In [ ]:
def _scalar_matmul_sum(a: np.ndarray, b: np.ndarray):
    n, k = a.shape
    m = b.shape[1]
    A = [[Value(a[i, p]) for p in range(k)] for i in range(n)]
    B = [[Value(b[p, j]) for j in range(m)] for p in range(k)]
    total = Value(0.0)
    for i in range(n):
        for j in range(m):
            acc = A[i][0] * B[0][j]
            for p in range(1, k):
                acc = acc + A[i][p] * B[p][j]
            total = total + acc
    return total, A


def _best_of(fn: Callable[[], Any], repeats: int = 3) -> float:
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


def _show_node_counts() -> None:
    n = 10
    rng = np.random.default_rng(SEED)
    a, b = rng.standard_normal((n, n)), rng.standard_normal((n, n))

    def run_scalar():
        total, A = _scalar_matmul_sum(a, b)
        total.backward()
        return total, A

    def run_tensor():
        ta, tb = Tensor(a), Tensor(b)
        total = (ta @ tb).sum()
        total.backward()
        return total, ta

    t_scalar, A = run_scalar()
    t_tensor, ta = run_tensor()
    grad_scalar = np.array([[A[i][p].grad for p in range(n)] for i in range(n)])
    print(f"sum of a {n} x {n} matrix product, gradient with respect to the first matrix")
    print(f"scalar engine: {len(topo_order(t_scalar))} nodes   tensor engine: "
          f"{len(topo_order(t_tensor))} nodes")
    err = relative_error(ta.grad, grad_scalar)
    print(f"the two gradients agree to a relative error below 1e-9: "
          f"{'yes' if err < 1e-9 else 'NO'}")
    s_time, t_time = _best_of(run_scalar), _best_of(run_tensor)
    print(f"scalar engine {1000 * s_time:.2f} ms, tensor engine {1000 * t_time:.3f} ms "
          f"(forward and backward, best of 3): the scalar engine is ≈{s_time / t_time:.0f}x "
          f"{'slower' if s_time > t_time else 'faster'} on this machine")


_try("node count demo", _show_node_counts,
     needs=("exercise 2", "exercise 3", "exercise 5", "exercise 6"))

## 8. Exercise 7 — a loss with no hand-derived backward: `softmax_cross_entropy()`

A classifier ends in softmax and cross-entropy: `softmax(z)_i = exp(z_i) / sum_j exp(z_j)`,
and the loss for one example with label `y` is `-log softmax(z)_y`, averaged over the batch.
Written literally, `exp(1000)` overflows to infinity and the loss becomes `nan`.

The fix is a property of softmax: subtracting the same constant from every logit in a row
does not change the result. Subtract the row maximum and every exponent is at most zero.
The maximum is a **constant** here, taken from `logits.data`, not a node in the graph: the
loss does not depend on it, so it must not carry gradient.

The point of this exercise is what you do **not** write: no derivative of softmax, no
derivative of the log. You compose `Tensor` operations you already trust, and `backward`
produces the gradient. A boolean mask built from the labels picks the logit of the true
class without any new operation.

<details><summary>💡 Hint 1 — what to think about</summary>

Write the loss as `shifted - log(sum(exp(shifted)))` for each row, where `shifted` is the
logits minus a constant column. Which of your operations keeps a column of shape `(n, 1)`
so that it broadcasts against `(n, c)`? How do you select entry `labels[i]` of row `i`
using only a product and a sum?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build a constant one-hot matrix from the labels (an identity matrix indexed by the labels
does it) and wrap it in a `Tensor`. Compute `shifted`, then the log-probabilities: `shifted`
minus the log of its row sums of exponentials. Multiply by the one-hot matrix, sum
everything, negate, and divide by the number of rows.
</details>

In [ ]:
def softmax_cross_entropy(logits: Tensor, labels: np.ndarray) -> Tensor:
    """Mean over the batch of `-log softmax(logits)[i, labels[i]]`, as a 0-d Tensor.

    `logits` is an `(n, c)` Tensor and `labels` an integer array of length n. Build the loss
    from Tensor operations only, so `loss.backward()` fills `logits.grad`. It must stay finite
    for logits of magnitude 1000. Do not modify `labels`.

    Example:
        >>> loss = softmax_cross_entropy(Tensor([[0.0, 0.0]]), np.array([0]))
        >>> float(loss.data)              # -log(1/2)
        0.6931471805599453
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_softmax_cross_entropy() -> None:
    rng = np.random.default_rng(SEED + 1)
    z = rng.standard_normal((5, 4))
    y = np.array([0, 3, 1, 1, 2])
    y_before = y.copy()
    loss = softmax_cross_entropy(Tensor(z), y)
    assert isinstance(loss, Tensor) and loss.data.shape == (), (
        "return a Tensor holding ONE number (shape ()): the mean over the batch, not a vector"
    )
    shifted = z - z.max(axis=1, keepdims=True)
    ref = -(shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True)))[np.arange(5), y].mean()
    assert abs(float(loss.data) - ref) < 1e-10, (
        f"got {float(loss.data):.6f}, a plain numpy computation gives {ref:.6f}: average over "
        "the rows, and take -log of the probability of the TRUE class"
    )
    assert np.array_equal(y, y_before), "labels were modified"
    err = gradient_error(lambda t: softmax_cross_entropy(t, y), [z])
    assert err < TOL, (
        f"gradient check: worst relative error {err:.1e}, tolerance {TOL:.0e}. Build the loss "
        "from Tensor operations only: a step done on .data, in plain numpy, cuts the graph "
        "and the gradient stops there"
    )
    big = Tensor(np.array([[1000.0, 0.0, -1000.0], [-1000.0, 1000.0, 0.0]]))
    with np.errstate(all="ignore"):
        stable = softmax_cross_entropy(big, np.array([0, 1]))
        stable.backward()
    assert np.isfinite(stable.data) and np.all(np.isfinite(big.grad)), (
        "the loss or its gradient is not finite at logits of magnitude 1000: subtract each "
        "row's maximum before exponentiating"
    )
    shifted_loss = softmax_cross_entropy(Tensor(z + 7.0), y)
    assert abs(float(shifted_loss.data) - float(loss.data)) < 1e-9, (
        "adding a constant to every logit must not change the loss"
    )


_try("exercise 7", _check_softmax_cross_entropy, needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6"))

### The instability you avoided

Here is the literal formula on the same two logit rows used by the check, in plain numpy,
next to your stable loss. Nothing is hidden: numpy is allowed to overflow, and the cell says
what it got.

In [ ]:
def _show_stability() -> None:
    z = np.array([[1000.0, 0.0, -1000.0], [-1000.0, 1000.0, 0.0]])
    y = np.array([0, 1])
    with np.errstate(all="ignore"):
        probs = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
        naive = float(-np.log(probs[np.arange(2), y]).mean())
    stable = softmax_cross_entropy(Tensor(z), y)
    print(f"the literal formula on logits up to {z.max():.0f}: loss = {naive}  "
          f"({'finite' if np.isfinite(naive) else 'not a number'})")
    print(f"your softmax_cross_entropy on the same logits: loss = {abs(float(stable.data)):.1e}  "
          f"({'finite' if np.isfinite(stable.data) else 'not finite'}); the true class "
          "dominates in both rows, so the right answer is very close to zero")


_try("stability demo", _show_stability, needs=("exercise 7",))

## 9. Train a network with the engine you wrote

Everything below is given: a three-armed spiral dataset (generated here from a fixed seed),
a one-hidden-layer network built from your `Tensor` operations, and a plain gradient-descent
loop. The loop never derives a gradient. It calls `loss.backward()`, and the parameters
move. Before it trains, the cell checks the whole network's gradient against your oracle, to
the tolerance stated at the top: a framework you cannot trust to differentiate is a framework
you cannot trust to train.

In [ ]:
def make_spirals(per_class: int, classes: int, seed: int) -> tuple:
    """`classes` interleaved spirals, `per_class` points each: points `(n, 2)`, labels `(n,)`."""
    rng = np.random.default_rng(seed)
    pts = np.zeros((per_class * classes, 2))
    labels = np.zeros(per_class * classes, dtype=np.int64)
    for k in range(classes):
        rows = slice(k * per_class, (k + 1) * per_class)
        radius = np.linspace(0.05, 1.0, per_class)
        angle = np.linspace(k * 4.0, (k + 1) * 4.0, per_class) + 0.2 * rng.standard_normal(per_class)
        pts[rows] = np.c_[radius * np.sin(angle), radius * np.cos(angle)]
        labels[rows] = k
    return pts, labels


def mlp_logits(params: Sequence[Tensor], x: Tensor) -> Tensor:
    w1, b1, w2, b2 = params
    return (x @ w1 + b1).tanh() @ w2 + b2


def init_mlp(sizes: Sequence[int], seed: int) -> list:
    rng = np.random.default_rng(seed)
    (n_in, n_hid), n_out = sizes[:2], sizes[2]
    return [Tensor(rng.standard_normal((n_in, n_hid)) * 0.8), Tensor(np.zeros(n_hid)),
            Tensor(rng.standard_normal((n_hid, n_out)) * 0.5), Tensor(np.zeros(n_out))]


def train_mlp(pts: np.ndarray, labels: np.ndarray, hidden: int, steps: int, lr: float,
              seed: int) -> tuple:
    params = init_mlp((pts.shape[1], hidden, int(labels.max()) + 1), seed)
    x = Tensor(pts)
    losses = []
    for _ in range(steps):
        loss = softmax_cross_entropy(mlp_logits(params, x), labels)
        loss.backward()
        losses.append(float(loss.data))
        for p in params:
            p.data -= lr * p.grad
    correct = int((mlp_logits(params, x).data.argmax(axis=1) == labels).sum())
    return params, losses, correct


def _show_training() -> None:
    pts, labels = make_spirals(per_class=50, classes=3, seed=SEED)
    sub = slice(0, None, 15)                       # a small slice keeps the oracle cheap
    p0 = init_mlp((2, 6, 3), SEED)
    err = gradient_error(
        lambda w1, b1, w2, b2: softmax_cross_entropy(
            mlp_logits([w1, b1, w2, b2], Tensor(pts[sub])), labels[sub]),
        [t.data for t in p0])
    print(f"whole-network gradient check ({len(labels[sub])} points, hidden width 6), "
          f"tolerance {TOL:.0e}: {'passed' if err <= TOL else 'FAILED'}")
    steps = 400
    params, losses, correct = train_mlp(pts, labels, hidden=24, steps=steps, lr=0.5, seed=SEED)
    print(f"gradient descent, {steps} full-batch steps on {len(labels)} points, hidden width 24:")
    print(f"  loss {losses[0]:.2f} -> {losses[-1]:.2f}, "
          f"{correct} of {len(labels)} points classified correctly")
    print(f"  chance level would be {len(labels) // 3} of {len(labels)}")
    print("  the loss " + ("fell" if losses[-1] < losses[0] else "did NOT fall") +
          " over training, with every gradient from your engine")


_try("training demo", _show_training,
     needs=("exercise 1", "exercise 2", "exercise 5", "exercise 6", "exercise 7"))

## 10. Exercise 8 — measure the gradient layer by layer: `layer_grad_norms()`

A deep network multiplies one Jacobian per layer on the way back. If those factors are
typically smaller than one, the gradient reaching the first layers is tiny (it *vanishes*);
if larger, it blows up. Which one happens depends on the activation and on the scale of
the weights, and you can measure it directly: record the norm of each layer's weight gradient.

Build the network exactly as specified, run one forward and one backward pass with your
engine, and return the gradient norm of every weight matrix, input side first. The loss is
the mean of the squared output, a stand-in that gives every layer a gradient. `ACTIVATIONS`
names the nonlinearities to choose between. Sigmoid is among them on purpose: its derivative
`s * (1 - s)` is never more than a quarter, so each sigmoid layer can shrink the backward
signal unless the weights make up for it, while tanh's derivative reaches 1 and relu's is
exactly 1 wherever it is active. The table after the exercise measures what that does.

<details><summary>💡 Hint 1 — what to think about</summary>

The data and the weights must be drawn from the one generator in a fixed order, or two runs
with the same seed will not agree. After `backward`, where does each weight matrix keep its
gradient, and which norm of a matrix treats all its entries alike?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Seed one `np.random.default_rng`. Draw the input batch first, then each weight matrix in
layer order, scaling each by `gain / sqrt(width)`. Loop over the layers applying
`h @ W` and the chosen activation. Take the mean of `h * h`, call `backward`, and collect
`np.linalg.norm` of each weight Tensor's `grad`.
</details>

In [ ]:
ACTIVATIONS: dict[str, Callable[[Tensor], Tensor]] = {
    "tanh": lambda t: t.tanh(),
    "relu": lambda t: t.relu(),
    "sigmoid": lambda t: (1.0 + (-t).exp()) ** -1,
}


def layer_grad_norms(depth: int, width: int, activation: str, gain: float,
                     seed: int) -> list:
    """Frobenius norm of dLoss/dW for each of `depth` weight matrices, input side first.

    Draw from `rng = np.random.default_rng(seed)` in this order: the input batch
    `rng.standard_normal((32, width))`, then for each layer in order a matrix
    `gain * rng.standard_normal((width, width)) / sqrt(width)`. Each layer maps
    `h -> ACTIVATIONS[activation](h @ W)`. The loss is `(h * h).mean()` over the final output.

    Example:
        >>> norms = layer_grad_norms(4, 8, "tanh", 1.0, seed=0)
        >>> len(norms)                    # one norm per weight matrix
        4
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_layer_grad_norms() -> None:
    norms = layer_grad_norms(4, 8, "tanh", 1.0, seed=3)
    assert isinstance(norms, list) and len(norms) == 4 and all(isinstance(v, float) for v in norms), (
        f"got {type(norms).__name__} of length {len(norms)}: return a list of 4 Python floats, "
        "one per weight matrix"
    )
    assert all(v > 0.0 for v in norms), (
        "a gradient norm of zero or less: did the backward pass run, and did you take the "
        "norm of .grad rather than .data?"
    )
    rng = np.random.default_rng(3)
    x = rng.standard_normal((32, 8))
    ws = [1.0 * rng.standard_normal((8, 8)) / math.sqrt(8) for _ in range(4)]
    wt = [Tensor(w) for w in ws]

    def build(*mats):
        h = Tensor(x)
        for m in mats:
            h = (h @ m).tanh()
        return (h * h).mean()

    build(*wt).backward()
    want = [float(np.linalg.norm(w.grad)) for w in wt]
    assert np.allclose(norms, want, rtol=1e-9, atol=0), (
        f"got {[round(v, 6) for v in norms]}, expected {[round(v, 6) for v in want]} for the "
        "same draws: draw the input first, then the weights in layer order, scale by "
        "gain / sqrt(width), and list the INPUT side first"
    )
    relu = layer_grad_norms(4, 8, "relu", 1.0, seed=3)
    sigmoid = layer_grad_norms(4, 8, "sigmoid", 1.0, seed=3)
    assert not np.allclose(relu, norms, rtol=1e-3) and not np.allclose(sigmoid, norms, rtol=1e-3), (
        "the relu or sigmoid run matches the tanh run: pick the activation from "
        "ACTIVATIONS[activation]"
    )


_try("exercise 8", _check_layer_grad_norms, needs=("exercise 2", "exercise 5", "exercise 6"))

### Read the table

A deep stack of layers, one seed. For each activation and weight scale the table prints the
gradient norm at the first layer, at the last layer, and their ratio, with a verdict worked
out from that ratio. Every figure comes from your engine. Each activation runs at two weight
scales, because whether a gradient shrinks or grows depends on the weights as well as on
the activation; the line under the table lists, from the table itself, which settings shrink.

In [ ]:
def _show_vanishing() -> None:
    depth, width = 12, 32
    configs = [("sigmoid", 1.0), ("sigmoid", 4.0), ("tanh", 1.0), ("tanh", 3.0),
               ("relu", 1.0), ("relu", math.sqrt(2.0))]
    print(f"depth {depth}, width {width}: weight-gradient norm of the first and the last layer")
    print(f"{'activation':>10} {'gain':>5} {'first layer':>12} {'last layer':>12} "
          f"{'first/last':>11}  verdict")
    rows = []
    for act, gain in configs:
        norms = layer_grad_norms(depth, width, act, gain, seed=SEED)
        ratio = norms[0] / norms[-1]
        verdict = ("shrinks toward the input" if ratio < 1e-2
                   else "grows toward the input" if ratio > 10.0 else "roughly level")
        rows.append((act, gain, norms[0], norms[-1], ratio))
        print(f"{act:>10} {gain:5.2f} {norms[0]:12.2e} {norms[-1]:12.2e} {ratio:11.2e}  {verdict}")
    shrinking = [f"{r[0]} at gain {r[1]:.2f}" for r in rows if r[4] < 1e-2]
    print("shrinks toward the input by more than a factor of 100: "
          + (", ".join(shrinking) if shrinking else "none of these settings"))
    relu_ratios = [r[4] for r in rows if r[0] == "relu"]
    spread = (max(relu_ratios) - min(relu_ratios)) / min(relu_ratios)
    first = [r[2] for r in rows if r[0] == "relu"]
    print(f"\nrelu at two different weight scales: first/last agree to within {1e-6:.0e} "
          f"(relative): {'yes' if spread < 1e-6 else 'no'}, while the first-layer norm itself "
          f"changes by a factor of {max(first) / min(first):.1e}")
    if spread < 1e-6:
        print("relu(c * z) = c * relu(z) for c > 0, so rescaling every weight matrix rescales "
              "every layer's gradient by one common factor: the gain changes their absolute "
              "size, not the ratio between layers")
    else:
        print("the ratio moved with the gain, which relu's scaling property rules out: look at "
              "how layer_grad_norms scales the weights")


_try("vanishing gradient demo", _show_vanishing,
     needs=("exercise 2", "exercise 5", "exercise 6", "exercise 8"))

## 11. Optional: compare with PyTorch

If PyTorch is installed on the machine you are using, the cell below builds the same small
network in both libraries and prints how far apart the gradients are. If it is not, the cell
says so and nothing else in this lesson is affected. Nothing here downloads or installs
anything.

In [ ]:
def _show_torch_comparison() -> None:
    rng = np.random.default_rng(SEED)
    x, w, b = rng.standard_normal((5, 3)), rng.standard_normal((3, 4)), rng.standard_normal(4)
    y = np.array([0, 2, 1, 3, 0])
    mine = [Tensor(a) for a in (x, w, b)]
    softmax_cross_entropy((mine[0] @ mine[1]).tanh() + mine[2], y).backward()
    # Everything below touches only PyTorch. If it is missing, or installed but unusable here
    # (a build that cannot exchange arrays with this numpy, say), the cell prints one line and
    # stops: an optional comparison must never fail your notebook.
    try:
        import torch
    except ImportError:
        print("PyTorch is not installed here, so this optional comparison is skipped; nothing "
              "else in this lesson needs it.")
        return
    except Exception as exc:
        print(f"PyTorch is installed but could not be imported ({type(exc).__name__}), so this "
              "optional comparison is skipped; nothing else in this lesson needs it.")
        return
    try:
        theirs = [torch.tensor(a.tolist(), dtype=torch.float64, requires_grad=True)
                  for a in (x, w, b)]
        logits = torch.tanh(theirs[0] @ theirs[1]) + theirs[2]
        torch.nn.functional.cross_entropy(logits, torch.tensor(y.tolist(), dtype=torch.long)).backward()
        worst = max(float(np.abs(m.grad - np.array(t.grad.tolist())).max())
                    for m, t in zip(mine, theirs))
    except Exception as exc:
        print(f"PyTorch {getattr(torch, '__version__', '?')} raised {type(exc).__name__} in this "
              "optional comparison, so it is skipped; nothing else in this lesson needs it.")
        return
    print(f"torch {torch.__version__}: largest absolute difference between your gradients and "
          f"torch.autograd's, over every entry of three arrays: {worst:.1e}")


_try("torch comparison", _show_torch_comparison, needs=("exercise 2", "exercise 5", "exercise 6", "exercise 7"))

## What you built

A numerical gradient and a relative-error comparison you can hold any gradient to; a scalar
reverse-mode engine whose entire machinery is an iterative topological sort and a handful of
local rules; a forward-mode engine that gets the same derivatives by a different route, with
the pass counts to show when each is cheaper; and a tensor engine whose gradients undo
broadcasting and survive a softmax at extreme logits. Training the spiral network and
measuring layer-by-layer gradients needed no new derivation: you wrote forward code, and the
engine supplied every backward step.

Keep the `Tensor` class: it is a complete, small differentiation engine that other numpy
code can reuse in place of a fresh hand-derived backward pass.

## Common mistakes

* **`=` where `+=` belongs in a backward rule.** It passes every test on a graph where each
  value is used once, and fails the first time a value is reused.
* **Forgetting that a broadcast operand was used many times.** Its gradient is the *sum* of
  the copies' gradients, with the operand's own shape.
* **Checking gradients in single precision, or with a step that is too large or too small.**
  The step-size table is the reminder: there is a sweet spot, and it is not the smallest
  number you can type.
* **Checking only the gradient you wrote.** Check every input of the function: a bug in the
  gradient of the second argument never shows up in the first.
* **Calling `backward` on a graph and then trusting `grad` fields from an earlier call.**
  This engine resets the gradient of every node in the graph it sweeps, for exactly that
  reason; a tensor that is not in that graph keeps its old gradient, and an engine that
  accumulates across calls (as PyTorch's does) needs an explicit zeroing step.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## Self-check

Pick one answer for each, and check your reasoning against the engine you built rather
than against the options.

1. In `x * x + x`, the node `x` ends up with gradient `2x + 1`. Which property of the
   backward rules makes that come out right?
   - (a) Each `_backward` assigns to its inputs' `grad`, so the last writer wins
   - (b) The sweep visits `x` once and doubles its gradient afterwards
   - (c) Each `_backward` adds into its inputs' `grad`, so every use of `x` contributes
   - (d) The graph is traversed from the inputs towards the output

2. A bias of shape `(4,)` is added to a batch of shape `(32, 4)`. What is the gradient of the
   loss with respect to the bias?
   - (a) The sum of the 32 row gradients, a vector of shape `(4,)`
   - (b) The mean of the 32 row gradients, whatever the loss is
   - (c) The upstream gradient as it arrives, of shape `(32, 4)`
   - (d) The gradient of the first row only

3. A scalar loss depends on a very large number of inputs. Which statement about computing
   its full gradient is right?
   - (a) Forward mode needs one pass; reverse mode needs one per input
   - (b) Both modes need one pass per input
   - (c) Both modes need a single pass
   - (d) Reverse mode needs one forward evaluation and one backward sweep; forward mode needs
     one pass per input

4. Your engine and a central-difference check, in double precision with a sensible step,
   disagree by a relative error far above the tolerance on a smooth network. What is the most
   likely explanation?
   - (a) Ordinary floating-point rounding, nothing to investigate
   - (b) A bug in a backward rule, or in the check itself, worth hunting down
   - (c) The network is too deep for finite differences to be meaningful
   - (d) Central differences are always that inaccurate

5. A recursive topological sort works on small graphs and then raises an error on a graph
   recorded from a long loop. Why?
   - (a) Recursion is slower than a loop by a constant factor
   - (b) numpy arrays cannot be visited recursively
   - (c) Python limits how deep recursion can go, so a long chain exhausts it, while an
     explicit stack is limited only by memory
   - (d) A recursive sort visits every node twice

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_oracle, ()),
        ("exercise 2", _check_topo_order, ()),
        ("exercise 3", _check_value, ("exercise 2",)),
        ("exercise 4", _check_dual, ()),
        ("exercise 5", _check_unbroadcast, ()),
        ("exercise 6", _check_tensor, ("exercise 1", "exercise 2", "exercise 5")),
        ("exercise 7", _check_softmax_cross_entropy,
         ("exercise 1", "exercise 2", "exercise 5", "exercise 6")),
        ("exercise 8", _check_layer_grad_norms, ("exercise 2", "exercise 5", "exercise 6")),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.